# AML Transaction Detection — LightGBM + Graph Features

## Dependencies

Standard imports plus a few things worth flagging:
- **Polars** is used instead of pandas throughout — the dataset is large enough that lazy execution makes a real difference
- **Snap ML GFP** is the graph feature extraction tool used in the original AMLworld paper — it extracts structural features like fan-in/out, degree and cycle participation directly from the transaction network
- **gc** is called manually at a few points to avoid memory issues

Custom utilities in `utils.py`:
- `increment_run_gfp` — runs GFP in chunks so graph state carries over between them, which matters for cycle detection
- `temporal_split` — splits by proportion while preserving chronological order
- `f1_eval` — minority class F1 callback for LightGBM

In [5]:
import sys
import numpy
import polars
import snapml

print(sys.executable)
print("Dependencies loaded successfully")

/mnt/c/Users/Pooja/Downloads/GitAML/aml-detection/.venv/bin/python
Dependencies loaded successfully


In [6]:
import gc

import numpy as np
import polars as pl
from snapml import GraphFeaturePreprocessor

from config import DATA_PATH
from utils import increment_run_gfp, temporal_split, f1_eval

## Data Loading

The dataset is the **AMLworld HI-Small** variant — ~5M transactions with a ~1:1000 illicit ratio. Available on [Kaggle](https://www.kaggle.com/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml?select=HI-Small_Patterns.txt).

Loading from a pre-processed parquet file rather than the raw CSV for performance. 

In [7]:
path = DATA_PATH
trx = pl.scan_csv(path + '/HI-Small_Trans.csv')

In [8]:
rename_map = {
    'Timestamp': 'timestamp',
    'From Bank': 'from_bank',
    'Account': 'from_account',
    'To Bank': 'to_bank',
    'Account.1': 'to_account',
    'Account_duplicated_0': 'to_account',
    'Amount Received': 'amount_rcv',
    'Receiving Currency': 'currency_rcv',
    'Amount Paid': 'amount_paid',
    'Payment Currency': 'currency_paid',
    'Payment Format': 'payment_format',
    'Is Laundering': 'is_laundering',
    'Bank Name': 'bank_name', 
    'Bank ID': 'bank_id', 
    'Account Number': 'account_id', 
    'Entity ID': 'entity_id', 
    'Entity Name': 'entity_name'
}

# Rename columns to snake_case
existing = {k: v for k, v in rename_map.items() if k in trx.collect_schema().names()}
trx = trx.rename(existing)

In [9]:
# Load accounts file and assign a unique integer ID to each account
act = (
    pl.scan_csv(path + '/' + 'HI-Small_accounts.csv')
    .with_row_index(name='account_uid')
)

# Rename columns to snake_case
existing = {k: v for k, v in rename_map.items() if k in act.collect_schema().names()}
act = act.rename(existing)

# Entity name contains the type and an ID e.g. "Individual #1234" — extract the type only
act = act.with_columns(
    pl.col("entity_name")
      .str.split(" #")
      .list.first()
      .str.replace_all(" ", "_")
      .alias("entity_type")
)

In [10]:
# Join account metadata onto transactions for both sender and receiver
trx = (
    trx
    .join(
        act.select(['bank_id', 'account_id', 'account_uid', 'entity_type']), 
        left_on=['from_bank', 'from_account'], 
        right_on=['bank_id', 'account_id'], 
        how='left'
    )
    .rename({'account_uid': 'from_account_uid'})
    .join(
        act.select(['bank_id', 'account_id', 'account_uid']), 
        left_on=['to_bank', 'to_account'], 
        right_on=['bank_id', 'account_id'], 
        how='left')
    .rename({'account_uid': 'to_account_uid'})
)

In [11]:
del act
gc.collect()

30

In [12]:
# Parse timestamp, sort chronologically and convert to elapsed hours since first transaction
# transactionID is a sequential integer index — used later for graph based features
trx = (
    trx
    .with_columns(pl.col('timestamp').str.to_datetime(format='%Y/%m/%d %H:%M'))
    .sort('timestamp')
    .with_columns(
        ((pl.col('timestamp') - pl.col('timestamp').min()).dt.total_seconds() / 3600)
        .round(2)
        .alias('elapsed')
    )
    .with_row_index(name='transactionID')
)

In [13]:
# Normalise all amounts to USD using fixed exchange rates anchored to September 2022
exchange_rates_to_usd = {
    "Australian Dollar": 1.4965,
    "Bitcoin":           0.0000505,
    "Brazil Real":       5.2281,
    "Canadian Dollar":   1.3228,
    "Euro":              1.0042,
    "Mexican Peso":      19.9270,
    "Ruble":             60.1963,
    "Rupee":             79.8491,
    "Saudi Riyal":       3.7500,
    "Shekel":            3.4700,
    "Swiss Franc":       0.9741,
    "UK Pound":          0.8741,
    "US Dollar":         1.0000,
    "Yen":               143.0040,
    "Yuan":              6.9165,
}

trx = (
    trx
    .with_columns(
        pl.col("currency_rcv").replace(exchange_rates_to_usd).cast(pl.Float64).alias("rate_rcv"),
        pl.col("currency_paid").replace(exchange_rates_to_usd).cast(pl.Float64).alias("rate_paid"),
    )
    .with_columns(
        (pl.col("amount_rcv") / pl.col("rate_rcv")).alias("amount_rcv_usd"),
        (pl.col("amount_paid") / pl.col("rate_paid")).alias("amount_paid_usd"),
    )
    .drop("rate_rcv", "rate_paid")
)

In [10]:
# Ordinal encoding for currency columns
currency_encoding = {
    "Australian Dollar": 0,
    "Bitcoin":           1,
    "Brazil Real":       2,
    "Canadian Dollar":   3,
    "Euro":              4,
    "Mexican Peso":      5,
    "Ruble":             6,
    "Rupee":             7,
    "Saudi Riyal":       8,
    "Shekel":            9,
    "Swiss Franc":       10,
    "UK Pound":          11,
    "US Dollar":         12,
    "Yen":               13,
    "Yuan":              14,
}

trx = (
    trx
    .with_columns(
        pl.col("currency_rcv").replace(currency_encoding).alias("currency_rcv_enc"),
        pl.col("currency_paid").replace(currency_encoding).alias("currency_paid_enc"),
    )
)

In [14]:
# Ordinal encoding for entity type
entity_type_encoding = {
    'Direct': 1,
    'Individual': 2,
    'Country': 3,
    'Partnership': 4,
    'Sole_Proprietorship': 5,
    'Corporation': 6
}

trx = (
    trx
    .with_columns(
        pl.col("entity_type").replace(entity_type_encoding).cast(pl.Int8).alias("entity_type_enc")
    )
)

In [15]:
# Ordinal encoding for payment format
payment_format_encoding = {
    "ACH":          0,
    "Bitcoin":      1,
    "Cash":         2,
    "Cheque":       3,
    "Credit Card":  4,
    "Reinvestment": 5,
    "Wire":         6,
}

trx = trx.with_columns(
    pl.col("payment_format").replace(payment_format_encoding).cast(pl.Int8).alias("payment_format_enc")
)

In [16]:
# Flag transactions where the sent and received currencies differ
trx = trx.with_columns(
    (pl.col("currency_rcv") != pl.col("currency_paid")).cast(pl.Int8).alias("different_currency")
)

In [17]:
# Bin transaction amount into 5 brackets (in USD): <1k, 1k-5k, 5k-10k, 10k-50k, 50k+
trx = trx.with_columns(
    pl.when(pl.col("amount_paid_usd") < 1000).then(pl.lit(0))
    .when(pl.col("amount_paid_usd") < 5000).then(pl.lit(1))
    .when(pl.col("amount_paid_usd") < 10000).then(pl.lit(2))
    .when(pl.col("amount_paid_usd") < 50000).then(pl.lit(3))
    .otherwise(pl.lit(4))
    .alias("amount_bin")
)

In [18]:
# Extract hour of day and day of week from timestamp
trx = trx.with_columns(
    pl.col("timestamp").dt.hour().alias("hour_of_day"),
    pl.col("timestamp").dt.weekday().alias("day_of_week"),
)

In [19]:
gc.collect()

0

## Graph-Based Features

Structural features are extracted from the transaction network using the **Snap ML Graph Feature Preprocessor (GFP)** — the same tool used in the original AMLworld paper.

GFP treats transactions as directed edges in a graph and computes features for each edge based on the local subgraph structure within a configurable time window. All features are computed using only past transactions relative to each edge, preventing data leakage.

The following feature groups are extracted:

**Vertex statistics** — aggregates of amount and elapsed time over each account's historical transactions: fan, degree, ratio, maximum, median, variance and skew. These capture behavioural patterns at the account level.

**Fan-in / Fan-out** — counts how many distinct counterparties sent to (fan-in) or received from (fan-out) an account within a time window. Binned into degree brackets. High fan-in on a receiver is a core layering signal.

**Degree** — in/out degree of the source and target accounts within a time window, binned into brackets.

**Scatter-Gather** — detects more complex multi-hop patterns: funds splitting across many accounts then recombining (scatter-gather)

**Temp-Cycle / LC-Cycle** — time-constrained cycles and length-constrained cycles

Time windows, length, bins were tuned by maximising mutual information between each feature and the target label on the training set.

In [20]:
# Select the set of columns needed by GFP and convert to numpy
# GFP expects: [edge id, source vertex, destination vertex, timestamp, variable(s)]
X = (
    trx.lazy()
    .select(['transactionID', 'from_account_uid', 'to_account_uid', 'elapsed', 'amount_rcv_usd'])
    .collect()
    .to_numpy()
    .astype(float)
)

In [21]:
y = trx.lazy().select('is_laundering').collect().to_numpy().astype(float).ravel()

In [22]:
# Pattern labels are optional and are not included in the raw transaction CSV.
if "pattern" in trx.collect_schema().names():
    pattern = trx.select("pattern").collect().to_numpy().flatten()
    np.save("pattern.npy", pattern)
else:
    print("No pattern column found; skipping pattern.npy export.")

No pattern column found; skipping pattern.npy export.


In [23]:
params = {
    "num_threads": 8,
    "time_window": 24, # default lookback window in hours for vertex stats
    "vertex_stats": True,
    "vertex_stats_cols": [3, 4], # elapsed and amount columns in X
    "vertex_stats_feats": [0, 1, 2, 6, 7, 8, 9],  # fan, degree, ratio, max, median, var, skew

    "fan": True,
    "fan_bins": [2, 4], # bin thresholds: 1, 2-3, 4+
    "fan_tw": 12, # tuned via MI search

    "degree": True,
    "degree_bins": [2, 4], # tuned via MI search

    "scatter-gather": True,
    "scatter-gather_bins": [2],
    "scatter-gather_tw": 96,  # 4-day window to capture multi-day patterns

    "temp-cycle": True,
    "temp-cycle_tw": 96,
    "temp-cycle_bins": [2],

    "lc-cycle": True,
    "lc-cycle_tw": 96,
    "lc-cycle_len": 12, # captures cycles up to 12 hops / ~4 days
    "lc-cycle_bins": [2],
}

In [21]:
# Initialise GFP — always use transform(), not fit_transform() as it inserts edges twice
try:
    gfp = GraphFeaturePreprocessor()
    gfp.set_params(params)
    gfp_available = True
except AttributeError as exc:
    gfp = None
    gfp_available = False
    print(
        "Snap ML GFP is unavailable in this environment; "
        "graph features will be zero-filled. "
        f"Details: {exc}"
    )

In [22]:
# Run GFP in incremental chunks so graph state carries over between them
if gfp_available:
    X_enriched = increment_run_gfp(gfp, X, verbose=True)
else:
    # Preserve the GFP output shape so downstream feature assembly still works.
    X_enriched = np.concatenate(
        (X, np.zeros((X.shape[0], 55), dtype=float)),
        axis=1,
    )

Processing rows 0 to 250000...
Processing rows 250000 to 500000...
Processing rows 500000 to 750000...
Processing rows 750000 to 1000000...
Processing rows 1000000 to 1250000...
Processing rows 1250000 to 1500000...
Processing rows 1500000 to 1750000...
Processing rows 1750000 to 2000000...
Processing rows 2000000 to 2250000...
Processing rows 2250000 to 2500000...
Processing rows 2500000 to 2750000...
Processing rows 2750000 to 3000000...
Processing rows 3000000 to 3250000...
Processing rows 3250000 to 3500000...
Processing rows 3500000 to 3750000...
Processing rows 3750000 to 4000000...
Processing rows 4000000 to 4250000...
Processing rows 4250000 to 4500000...
Processing rows 4500000 to 4750000...
Processing rows 4750000 to 5000000...
Processing rows 5000000 to 5078345...


In [23]:
del X

In [27]:
np.save("x_enriched.npy", X_enriched) # Backup

In [24]:
X_enriched = np.load("x_enriched.npy")

In [26]:
currency_encoding = {
    "Australian Dollar": 0,
    "Bitcoin": 1,
    "Brazil Real": 2,
    "Canadian Dollar": 3,
    "Euro": 4,
    "Mexican Peso": 5,
    "Ruble": 6,
    "Rupee": 7,
    "Saudi Riyal": 8,
    "Shekel": 9,
    "Swiss Franc": 10,
    "UK Pound": 11,
    "US Dollar": 12,
    "Yen": 13,
    "Yuan": 14,
}

trx = trx.with_columns(
    pl.col("currency_rcv")
      .replace(currency_encoding)
      .cast(pl.Int8)
      .alias("currency_rcv_enc"),
    pl.col("currency_paid")
      .replace(currency_encoding)
      .cast(pl.Int8)
      .alias("currency_paid_enc"),
)

In [28]:
# Pull the preset features into a numpy array to concatenate with GFP output later
cols = [
    'currency_rcv_enc',
    'currency_paid_enc',
    'entity_type_enc',
    'payment_format_enc',
    'different_currency',
    'amount_bin',
    'hour_of_day',
    'day_of_week'
]

X_preset_features = (
    trx.lazy()
    .select(cols)
    .collect()
    .to_numpy()
    .astype(float)
)

In [29]:
del trx

In [30]:
# Concatenate GFP features with preset features
X_combined = np.concatenate((X_enriched, X_preset_features), axis=1)

In [31]:
del X_preset_features, X_enriched

In [32]:
feature_names = [
    "fan-in 2-4",                    # 5
    "fan-in 4+",                     # 6
    "fan-out 2-4",                   # 7
    "fan-out 4+",                    # 8
    "degree-in 2-4",                 # 9
    "degree-in 4+",                  # 10
    "degree-out 2-4",                # 11
    "degree-out 4+",                 # 12
    "scatter-gather 2+",             # 13
    "temporal-cycle 2+",             # 14
    "lc-cycle 2+",                   # 15
    "src-vertex-fan-out",            # 16
    "src-vertex-fan-in",             # 17
    "dst-vertex-fan-out",            # 18
    "dst-vertex-fan-in",             # 19
    "src-vertex-degree-out",         # 20
    "src-vertex-degree-in",          # 21
    "dst-vertex-degree-out",         # 22
    "dst-vertex-degree-in",          # 23
    "src-vertex-ratio-out",          # 24
    "src-vertex-ratio-in",           # 25
    "dst-vertex-ratio-out",          # 26
    "dst-vertex-ratio-in",           # 27
    "src-vertex-out-max-time",       # 28
    "src-vertex-out-max-amount",     # 29
    "src-vertex-in-max-time",        # 30
    "src-vertex-in-max-amount",      # 31
    "dst-vertex-out-max-time",       # 32
    "dst-vertex-out-max-amount",     # 33
    "dst-vertex-in-max-time",        # 34
    "dst-vertex-in-max-amount",      # 35
    "src-vertex-out-median-time",    # 36
    "src-vertex-out-median-amount",  # 37
    "src-vertex-in-median-time",     # 38
    "src-vertex-in-median-amount",   # 39
    "dst-vertex-out-median-time",    # 40
    "dst-vertex-out-median-amount",  # 41
    "dst-vertex-in-median-time",     # 42
    "dst-vertex-in-median-amount",   # 43
    "src-vertex-out-var-time",       # 44
    "src-vertex-out-var-amount",     # 45
    "src-vertex-in-var-time",        # 46
    "src-vertex-in-var-amount",      # 47
    "dst-vertex-out-var-time",       # 48
    "dst-vertex-out-var-amount",     # 49
    "dst-vertex-in-var-time",        # 50
    "dst-vertex-in-var-amount",      # 51
    "src-vertex-out-skew-time",      # 52
    "src-vertex-out-skew-amount",    # 53
    "src-vertex-in-skew-time",       # 54
    "src-vertex-in-skew-amount",     # 55
    "dst-vertex-out-skew-time",      # 56
    "dst-vertex-out-skew-amount",    # 57
    "dst-vertex-in-skew-time",       # 58
    "dst-vertex-in-skew-amount",     # 59
    "currency_rcv_enc",              # 60
    "currency_paid_enc",             # 61
    "entity_type_enc",               # 62
    "payment_format_enc",            # 63
    "different_currency",            # 64
    "amount_bin",                    # 65
    "hour_of_day",                   # 66
    "day_of_week",                   # 67
]

In [33]:
# Drop low-importance features identified during initial LightGBM runs
# fan-out 2-4 (7), scatter-gather 2+ (13), src-vertex-in-median-time (38)
cols_to_drop = [7, 13, 38]
cols_to_keep = [i for i in range(X_combined.shape[1]) if i not in cols_to_drop]
X_filtered = X_combined[:, cols_to_keep]

In [34]:
del X_combined
gc.collect()

287

In [35]:
# Stack target column onto the feature array for easy splitting later
X_stacked = np.hstack([X_filtered, y.reshape(-1, 1)])

In [36]:
# Save to disk — avoids rerunning the full GFP pipeline on subsequent runs
np.save('transaction_data.npy', X_stacked)